# Building a Simple AI Agent: Weather Assistant using Tavily Search

### A beginner-friendly tutorial

In this notebook, we will build a small **AI agent** that can answer questions about the weather by:

1. Using the **Tavily Search API** to fetch live, up-to-date weather information from the web.
2. Using a **free, open-source Large Language Model (LLM)** to understand the user's question, decide to use the weather tool, and turn the search results into a friendly answer.

### What is an "agent"?

A normal chatbot just replies using what it already knows. An **agent** is a chatbot that can also:

- Look at the tools it has available (in our case, a weather search tool).
- Decide **when** it needs to use a tool.
- Use the tool, read the result, and then write a final answer.

Think of it like a human assistant who, instead of guessing today's weather, opens a weather app, checks it, and then tells you the answer.

### What we will use

| Piece | Purpose | Cost |
|---|---|---|
| **Tavily API** | A search API built for AI agents — great at fetching current, factual information (like today's weather) | Free tier available |
| **Ollama + an open-source model** (e.g. `llama3`) | The "brain" of our agent — completely free and runs on your own computer | Free (open source) |
| **LangChain** | A Python framework that makes it easy to connect an LLM to tools and build an agent | Free (open source) |

By the end, you'll have a working agent you can ask things like *"What's the weather in Tokyo right now?"*

## Step 0: Prerequisites (do this before running the code)

1. **Get a free Tavily API key**
   - Go to https://tavily.com and sign up (free tier is enough for this demo).
   - Copy your API key from the dashboard — it usually looks like `tvly-xxxxxxxxxxxxxxxxxxxx`.

2. **Install Ollama (to run the open-source model locally, for free)**
   - Download it from https://ollama.com for your operating system.
   - After installing, open a terminal and run:
     ```bash
     ollama pull llama3
     ```
     This downloads the open-source `llama3` model to your machine (a few GB, one-time download).
   - Keep Ollama running in the background — it works like a small local server.

   > **No GPU? No problem.** Smaller open models like `llama3.2` or `phi3` also work well and are lighter — just replace `llama3` with `llama3.2` anywhere in this notebook.

3. **Never share or hardcode your API key.** We will store it as an **environment variable** instead — explained in Step 2 below.

## Step 1: Install the required Python packages

- `tavily-python` → the official Tavily client
- `langchain`, `langchain-community`, `langchain-ollama` → to build the agent and connect it to our open-source model
- `python-dotenv` → to safely load our API key from a `.env` file

In [ ]:
!pip install -q tavily-python langchain langchain-community langchain-ollama python-dotenv

## Step 2: Using your Tavily API key as an environment variable

**Why an environment variable and not just pasting the key into the code?**

If you paste your API key directly into a notebook, it's easy to accidentally share it (e.g. by uploading the notebook to GitHub) and expose your account. Storing it as an environment variable keeps the secret **out of your code**.

There are two common ways to do this. Pick whichever is easier for you.

### Option A — Using a `.env` file (recommended, most beginner-friendly)

1. In the same folder as this notebook, create a new file named exactly `.env`
2. Open it in a text editor and add this single line (replace with your real key):
   ```
   TAVILY_API_KEY=tvly-your-real-key-here
   ```
3. Save the file. That's it — the code below will automatically read it.

> **Tip:** If you're using git, add `.env` to your `.gitignore` file so it never gets uploaded anywhere.

### Option B — Setting it directly in your terminal (temporary, for the current session only)

- **Mac/Linux:**
  ```bash
  export TAVILY_API_KEY="tvly-your-real-key-here"
  ```
- **Windows (PowerShell):**
  ```powershell
  $env:TAVILY_API_KEY="tvly-your-real-key-here"
  ```
  Then launch Jupyter from that same terminal window.

### Option C — Entering it securely just for this notebook session

If you don't want to create a `.env` file right now, the code cell below will safely **prompt you to type the key** (hidden, like a password) and set it as an environment variable just for this session — nothing is saved to disk.

In [ ]:
import os
from dotenv import load_dotenv
import getpass

# Try to load variables from a .env file, if one exists in this folder
load_dotenv()

# If TAVILY_API_KEY wasn't found in the environment or .env file, ask for it securely
if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

# Quick check (never print the full key!)
key = os.environ.get("TAVILY_API_KEY", "")
if key:
    print(f"Tavily API key loaded successfully. Starts with: {key[:8]}...")
else:
    print("No Tavily API key found. Please set it before continuing.")

## Step 3: Test the Tavily API directly

Before wiring it into an agent, let's confirm the Tavily API key works by making a simple search call.

In [ ]:
from tavily import TavilyClient

# The client automatically reads the TAVILY_API_KEY environment variable
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

test_results = tavily_client.search("current weather in London", max_results=3)

for r in test_results["results"]:
    print(r["title"])
    print(r["content"][:200], "...")
    print("-" * 60)

## Step 4: Connect to a free, open-source LLM (via Ollama)

`ChatOllama` connects to the `llama3` model running locally through Ollama (started with `ollama pull llama3`, as described in Step 0). Nothing here is sent to any paid API — the model runs entirely on your own machine.

In [ ]:
from langchain_ollama import ChatOllama

# You can swap "llama3" for any other open-source model you've pulled with Ollama,
# e.g. "llama3.2", "mistral", or "phi3"
llm = ChatOllama(model="llama3", temperature=0)

# Quick sanity check
response = llm.invoke("Say hello in one short sentence.")
print(response.content)

## Step 5: Turn Tavily search into a "Weather Tool" for the agent

An agent needs its tools defined clearly, with a name and description, so the LLM knows *when* and *how* to use them. Below we wrap Tavily's search specifically for weather questions.

In [ ]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city by searching the web. Input should be a city name, e.g. 'Paris' or 'New York'."""
    query = f"current weather in {city} today"
    results = tavily_client.search(query, max_results=3)

    combined = ""
    for r in results["results"]:
        combined += f"- {r['title']}: {r['content'][:300]}\n"

    return combined if combined else "No weather information found."

## Step 6: Build the agent

Now we give the LLM access to our `get_weather` tool. LangChain's agent will:

1. Read the user's question.
2. Decide whether it needs the weather tool.
3. Call the tool if needed.
4. Use the tool's result to write a natural-language answer.

In [ ]:
from langchain.agents import initialize_agent, AgentType

tools = [get_weather]

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,        # verbose=True lets you see the agent's step-by-step reasoning
    handle_parsing_errors=True,
)

## Step 7: Ask the agent about the weather

Run the cell below and watch the printed output — because `verbose=True`, you'll see the agent's *thoughts*, which tool it decides to call, and the final answer it gives you.

In [ ]:
question = "What's the weather like in Tokyo right now? Should I bring an umbrella?"
answer = agent.invoke({"input": question})
print("\nFinal answer:\n", answer["output"])

In [ ]:
# Try another one!
question2 = "Compare the weather in Paris and New York today."
answer2 = agent.invoke({"input": question2})
print("\nFinal answer:\n", answer2["output"])

## Notes, tips & troubleshooting

- **`Connection refused` / Ollama errors:** Make sure the Ollama app is running in the background and that you've run `ollama pull llama3` at least once.
- **Tavily errors about the API key:** Double-check your `.env` file has no extra spaces or quotes around the key, and that it's saved in the same folder as this notebook.
- **Want to use a different open-source model?** Any model available via `ollama pull <model-name>` works — just update the `model=` parameter in Step 4.
- **Prefer not to install Ollama?** You can use Hugging Face's free Inference API instead with `langchain_huggingface.HuggingFaceEndpoint`, using an open model such as `meta-llama/Meta-Llama-3-8B-Instruct` and a free Hugging Face access token (stored as the `HUGGINGFACEHUB_API_TOKEN` environment variable, exactly the same way we handled `TAVILY_API_KEY` above).
- **`verbose=True`** is very useful while learning — it shows you exactly how the agent "thinks." You can set it to `False` once you're comfortable and just want the final answer.

## Summary

In this notebook you learned how to:

1. Store an API key safely as an **environment variable** instead of hardcoding it.
2. Use the **Tavily API** to fetch live information from the web.
3. Run a **free, open-source LLM** locally using Ollama.
4. Wrap a function as a **tool** and give it to an LLM.
5. Build a simple **agent** that decides when to use that tool and answers naturally.

From here, you could extend this agent with more tools (news search, currency conversion, a calculator, etc.) — the pattern stays exactly the same: define a tool, add it to the `tools` list, and the agent will learn to use it.